# Neo4j Graph Storage with LlamaIndex Tutorial

This tutorial demonstrates how to use Neo4j as a graph storage backend with LlamaIndex to create and query knowledge graphs from documents.

## What you'll learn:
- Setting up Neo4j graph storage with LlamaIndex
- Creating knowledge graphs from documents
- Querying graph data using graph-based retrieval
- Understanding relationships between entities

## Prerequisites:
- Neo4j database (local or cloud instance)
- OpenAI API key for embeddings and LLM
- Python environment with required packages

## Installation

First, let's install the required packages for Neo4j graph storage with LlamaIndex:

In [ ]:
# Install required packages
%pip install llama-index-graph-stores-neo4j
%pip install llama-index-llms-openai
%pip install llama-index-embeddings-openai
%pip install neo4j
%pip install python-dotenv

## Setting up Neo4j

### Option 1: Local Neo4j Setup
If you want to run Neo4j locally, you can:
1. Download Neo4j Desktop from https://neo4j.com/download/
2. Create a new database instance
3. Start the database and note the connection details

### Option 2: Neo4j AuraDB (Cloud)
1. Sign up at https://console.neo4j.io/
2. Create a free AuraDB instance
3. Get your connection URI and credentials

### Option 3: Docker
```bash
docker run \
    --name neo4j \
    -p7474:7474 -p7687:7687 \
    -d \
    -e NEO4J_AUTH=neo4j/password \
    neo4j:latest
```

## Environment Setup

Create a `.env` file in your project directory with the following variables:

```
OPENAI_API_KEY=your_openai_api_key_here
NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=password
```

In [ ]:
import os
import nest_asyncio
from dotenv import load_dotenv
from llama_index.core import Document, Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.graph_stores.neo4j import Neo4jGraphStore
from llama_index.core import KnowledgeGraphIndex, StorageContext
from llama_index.core.graph_stores import SimpleGraphStore

# Enable nested async calls
nest_asyncio.apply()

# Load environment variables
load_dotenv()

# Set up OpenAI credentials
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Please set your OPENAI_API_KEY in the .env file")

# Set up Neo4j credentials
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

print("Environment variables loaded successfully!")
print(f"Neo4j URI: {NEO4J_URI}")
print(f"Neo4j Username: {NEO4J_USERNAME}")

## Configure LlamaIndex Settings

Set up the global settings for LLM and embeddings:

In [ ]:
# Configure LlamaIndex settings
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

print("LlamaIndex settings configured!")

## Initialize Neo4j Graph Store

Now let's connect to Neo4j and create our graph store:

In [ ]:
try:
    # Initialize Neo4j graph store
    graph_store = Neo4jGraphStore(
        uri=NEO4J_URI,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD,
        database="neo4j"
    )
    
    print("Successfully connected to Neo4j!")
    
    # Test the connection
    # Note: This will depend on the actual Neo4j graph store implementation
    print("Neo4j graph store initialized successfully!")
    
except Exception as e:
    print(f"Failed to connect to Neo4j: {e}")
    print("Using SimpleGraphStore as fallback...")
    graph_store = SimpleGraphStore()
    print("SimpleGraphStore initialized for demonstration purposes.")

## Prepare Sample Documents

Let's create some sample documents about technology and relationships that will form interesting graph structures:

In [ ]:
# Sample documents for creating a knowledge graph
documents = [
    Document(
        text="""
        Apple Inc. is a technology company founded by Steve Jobs, Steve Wozniak, and Ronald Wayne in 1976.
        The company is headquartered in Cupertino, California. Apple is known for products like the iPhone,
        iPad, Mac computers, and Apple Watch. Tim Cook is the current CEO of Apple, having taken over
        from Steve Jobs in 2011.
        """,
        metadata={"source": "tech_companies", "topic": "Apple"}
    ),
    Document(
        text="""
        Microsoft Corporation was founded by Bill Gates and Paul Allen in 1975. The company is based in
        Redmond, Washington. Microsoft develops Windows operating system, Office productivity suite,
        Azure cloud services, and Xbox gaming console. Satya Nadella is the current CEO of Microsoft,
        having succeeded Steve Ballmer in 2014.
        """,
        metadata={"source": "tech_companies", "topic": "Microsoft"}
    ),
    Document(
        text="""
        Google was founded by Larry Page and Sergey Brin in 1998 while they were PhD students at
        Stanford University. The company is headquartered in Mountain View, California. Google's
        parent company is Alphabet Inc. Sundar Pichai is the CEO of Google and Alphabet.
        Google's main products include Search, Gmail, Chrome browser, Android OS, and YouTube.
        """,
        metadata={"source": "tech_companies", "topic": "Google"}
    ),
    Document(
        text="""
        The iPhone was first released by Apple in 2007, revolutionizing the smartphone industry.
        It runs on iOS operating system and competes with Android-based smartphones from companies
        like Samsung and Google. The iPhone uses Apple's A-series processors and integrates with
        other Apple products through the Apple ecosystem.
        """,
        metadata={"source": "products", "topic": "iPhone"}
    ),
    Document(
        text="""
        Steve Jobs was born in 1955 and co-founded Apple Computer Company in 1976. He left Apple in 1985
        and founded NeXT Computer. He also purchased Pixar Animation Studios from Lucasfilm in 1986.
        Jobs returned to Apple in 1997 and led the company's renaissance with products like the iMac,
        iPod, iPhone, and iPad. He passed away in 2011.
        """,
        metadata={"source": "biography", "topic": "Steve Jobs"}
    )
]

print(f"Created {len(documents)} sample documents for the knowledge graph.")
for i, doc in enumerate(documents):
    print(f"Document {i+1}: {doc.metadata['topic']} ({len(doc.text.strip())} characters)")

## Create Knowledge Graph Index

Now let's create a knowledge graph from our documents:

In [ ]:
# Create storage context with our graph store
storage_context = StorageContext.from_defaults(graph_store=graph_store)

# Create knowledge graph index
print("Creating knowledge graph index... This may take a few moments.")

kg_index = KnowledgeGraphIndex.from_documents(
    documents,
    storage_context=storage_context,
    max_triplets_per_chunk=10,  # Maximum number of knowledge triplets to extract per chunk
    include_embeddings=True,    # Include embeddings for similarity search
    show_progress=True          # Show progress during index creation
)

print("Knowledge graph index created successfully!")

## Explore the Knowledge Graph

Let's examine what triplets (subject-predicate-object relationships) were extracted:

In [ ]:
# Get all the triplets from the graph
try:
    triplets = graph_store.get_triplets()
    print(f"Total triplets in knowledge graph: {len(triplets)}")
    print("\nSample triplets:")
    for i, triplet in enumerate(triplets[:10]):  # Show first 10 triplets
        print(f"{i+1}. {triplet}")
        
    if len(triplets) > 10:
        print(f"... and {len(triplets) - 10} more triplets")
        
except Exception as e:
    print(f"Could not retrieve triplets directly: {e}")
    print("This is normal for some graph store implementations.")

## Query the Knowledge Graph

Now let's create a query engine and ask questions about our knowledge graph:

In [ ]:
# Create query engine for the knowledge graph
kg_query_engine = kg_index.as_query_engine(
    include_text=True,          # Include original text in responses
    response_mode="tree_summarize",  # How to synthesize responses
    embedding_mode="hybrid",    # Use both embeddings and graph structure
    similarity_top_k=3          # Number of similar nodes to retrieve
)

print("Knowledge graph query engine created!")

In [ ]:
# Query 1: Ask about relationships
query1 = "Who founded Apple and what is their relationship to the company?"
print(f"Query: {query1}")
print("=" * 50)

response1 = kg_query_engine.query(query1)
print(f"Response: {response1}")
print("\n")

In [ ]:
# Query 2: Ask about connections between entities
query2 = "What are the relationships between Steve Jobs, Apple, and other companies?"
print(f"Query: {query2}")
print("=" * 50)

response2 = kg_query_engine.query(query2)
print(f"Response: {response2}")
print("\n")

In [ ]:
# Query 3: Ask about tech company comparisons
query3 = "Compare the founding stories of Apple, Microsoft, and Google. What are the similarities and differences?"
print(f"Query: {query3}")
print("=" * 50)

response3 = kg_query_engine.query(query3)
print(f"Response: {response3}")
print("\n")

In [ ]:
# Query 4: Ask about product relationships
query4 = "Tell me about the iPhone and its relationship to Apple's ecosystem."
print(f"Query: {query4}")
print("=" * 50)

response4 = kg_query_engine.query(query4)
print(f"Response: {response4}")
print("\n")

## Advanced: Graph-specific Queries

Let's try some queries that specifically leverage the graph structure:

In [ ]:
# Graph-specific query engine that focuses more on relationships
graph_query_engine = kg_index.as_query_engine(
    include_text=False,         # Focus on graph structure rather than text
    embedding_mode="graph",     # Use only graph structure
    similarity_top_k=5
)

# Query for entity relationships
graph_query = "What entities are connected to Steve Jobs?"
print(f"Graph Query: {graph_query}")
print("=" * 50)

graph_response = graph_query_engine.query(graph_query)
print(f"Graph Response: {graph_response}")

## Working with Real Document Data

Let's also try with one of the PDF files in the data directory:

In [ ]:
from llama_index.core import SimpleDirectoryReader
import os

# Check if data directory exists and load a document
data_dir = "./data"
if os.path.exists(data_dir):
    print(f"Loading documents from {data_dir}...")
    
    # Load documents from data directory
    loader = SimpleDirectoryReader(data_dir)
    real_documents = loader.load_data()
    
    if real_documents:
        print(f"Loaded {len(real_documents)} documents from data directory")
        
        # Take only the first document to avoid processing too much data
        sample_doc = real_documents[0]
        print(f"Sample document: {sample_doc.metadata.get('file_name', 'Unknown')}")
        print(f"Document length: {len(sample_doc.text)} characters")
        
        # Create a smaller knowledge graph from the real document
        print("\nCreating knowledge graph from real document...")
        
        # Split the document into smaller chunks for better processing
        chunk_size = 2000  # Limit chunk size for demo
        if len(sample_doc.text) > chunk_size:
            sample_doc.text = sample_doc.text[:chunk_size] + "..."
        
        # Create new graph store for real document
        real_storage_context = StorageContext.from_defaults(graph_store=SimpleGraphStore())
        
        real_kg_index = KnowledgeGraphIndex.from_documents(
            [sample_doc],
            storage_context=real_storage_context,
            max_triplets_per_chunk=5,  # Fewer triplets for demo
            include_embeddings=True,
            show_progress=True
        )
        
        # Query the real document graph
        real_query_engine = real_kg_index.as_query_engine()
        
        # Ask a question about the document
        doc_query = "What are the main entities and their relationships in this document?"
        print(f"\nQuerying real document: {doc_query}")
        doc_response = real_query_engine.query(doc_query)
        print(f"Response: {doc_response}")
        
    else:
        print("No documents found in data directory")
else:
    print(f"Data directory {data_dir} not found")

## Cleanup and Best Practices

When working with Neo4j in production, remember to:

In [ ]:
# Clean up (optional) - be careful with this in production!
# This will clear the entire graph database

def cleanup_graph(confirm=False):
    """Clean up the graph database - use with caution!"""
    if confirm:
        try:
            # This would clear all nodes and relationships
            # graph_store.clear()  # Uncomment if this method exists
            print("Graph database cleaned up")
        except Exception as e:
            print(f"Cleanup failed: {e}")
    else:
        print("Cleanup skipped - set confirm=True to actually clean up")

# Demonstrate cleanup (but don't actually do it)
cleanup_graph(confirm=False)

## Summary

In this tutorial, you learned how to:

1. **Set up Neo4j graph storage** with LlamaIndex
2. **Create knowledge graphs** from documents by extracting entities and relationships
3. **Query the graph** using natural language to find connections and relationships
4. **Compare different query modes** (hybrid vs. graph-only)
5. **Work with real documents** from PDFs

### Key Benefits of Graph Storage:
- **Relationship Discovery**: Automatically discover connections between entities
- **Complex Queries**: Answer questions about relationships and connections
- **Knowledge Representation**: Represent domain knowledge as interconnected concepts
- **Scalability**: Neo4j can handle large, complex graphs efficiently

### Next Steps:
- Experiment with larger document collections
- Customize the knowledge extraction process
- Combine graph queries with vector similarity search
- Explore Neo4j's native query language (Cypher) for advanced graph operations
- Implement graph visualization to explore your knowledge graphs visually

### Resources:
- [LlamaIndex Documentation](https://docs.llamaindex.ai/)
- [Neo4j Documentation](https://neo4j.com/docs/)
- [Neo4j Graph Academy](https://graphacademy.neo4j.com/)
- [Cypher Query Language](https://neo4j.com/developer/cypher/)
